In [1]:
# ===== CELL 1: SETUP =====
import pandas as pd
import sqlite3
import os

PROCESSED_DIR = "data/processed"
DB_PATH = "data/startup_intelligence.db"

print("Files available in processed:", os.listdir(PROCESSED_DIR))

Files available in processed: ['ref_real_unicorns.csv', 'ml_training_set.csv', 'fact_funding_rounds.csv', 'dim_startup.csv', 'supplementary_funding_events.csv', 'dim_investor.csv']


In [2]:
# ===== CELL 2: LOAD PROCESSED DATA INTO MEMORY =====
df_startups = pd.read_csv(os.path.join(PROCESSED_DIR, "dim_startup.csv"))
df_investors = pd.read_csv(os.path.join(PROCESSED_DIR, "dim_investor.csv"))
df_funding = pd.read_csv(os.path.join(PROCESSED_DIR, "fact_funding_rounds.csv"))
df_unicorns_real = pd.read_csv(os.path.join(PROCESSED_DIR, "ref_real_unicorns.csv"))
df_supplementary = pd.read_csv(os.path.join(PROCESSED_DIR, "supplementary_funding_events.csv"))
df_ml_training = pd.read_csv(os.path.join(PROCESSED_DIR, "ml_training_set.csv"))

print("All processed tables loaded:")
for name in ['df_startups', 'df_investors', 'df_funding', 'df_unicorns_real', 'df_supplementary', 'df_ml_training']:
    print(f"  {name}: {eval(name).shape}")

All processed tables loaded:
  df_startups: (1747, 10)
  df_investors: (73, 11)
  df_funding: (2818, 19)
  df_unicorns_real: (1360, 7)
  df_supplementary: (4847, 6)
  df_ml_training: (100000, 11)


In [3]:
# ===== CELL 3: CREATE / CONNECT TO SQLITE DATABASE =====
os.makedirs("data", exist_ok=True)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"Connected to {DB_PATH}")

Connected to data/startup_intelligence.db


In [4]:
# ===== CELL 4: CREATE TABLE SCHEMA =====

cursor.executescript("""
DROP TABLE IF EXISTS fact_funding_rounds;
DROP TABLE IF EXISTS dim_startup;
DROP TABLE IF EXISTS dim_investor;
DROP TABLE IF EXISTS ref_real_unicorns;
DROP TABLE IF EXISTS supplementary_funding_events;
DROP TABLE IF EXISTS ml_training_set;

CREATE TABLE dim_startup (
    startup_id TEXT PRIMARY KEY,
    startup_name TEXT,
    sector TEXT,
    country TEXT,
    region TEXT,
    city TEXT,
    founded_year INTEGER,
    employee_count INTEGER,
    female_founder INTEGER,
    revenue_status TEXT
);

CREATE TABLE dim_investor (
    investor_id TEXT PRIMARY KEY,
    investor_name TEXT,
    investor_type TEXT,
    hq_country TEXT,
    founded_year INTEGER,
    aum_billion_usd REAL,
    total_deals_in_dataset INTEGER,
    total_invested_million_usd REAL,
    avg_deal_size_million_usd REAL,
    top_sector TEXT,
    top_stage TEXT
);

CREATE TABLE fact_funding_rounds (
    round_id TEXT PRIMARY KEY,
    startup_id TEXT,
    startup_name TEXT,
    funding_date TEXT,
    year INTEGER,
    quarter TEXT,
    funding_stage TEXT,
    amount_million_usd REAL,
    pre_money_valuation_million_usd REAL,
    post_money_valuation_million_usd REAL,
    equity_dilution_pct REAL,
    lead_investor TEXT,
    investor_type TEXT,
    num_investors INTEGER,
    sector TEXT,
    country TEXT,
    region TEXT,
    city TEXT,
    investor_id TEXT,
    FOREIGN KEY (startup_id) REFERENCES dim_startup(startup_id),
    FOREIGN KEY (investor_id) REFERENCES dim_investor(investor_id)
);

CREATE TABLE ref_real_unicorns (
    unicorn_id INTEGER PRIMARY KEY AUTOINCREMENT,
    company TEXT,
    valuation_billion_usd REAL,
    date_joined TEXT,
    country TEXT,
    city TEXT,
    industry TEXT,
    select_investors TEXT
);

CREATE TABLE supplementary_funding_events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    startup_name TEXT,
    industry TEXT,
    country TEXT,
    funding_amount_usd REAL,
    funding_stage TEXT,
    source_dataset TEXT
);

CREATE TABLE ml_training_set (
    record_id INTEGER PRIMARY KEY AUTOINCREMENT,
    funding_rounds INTEGER,
    founder_experience_years INTEGER,
    team_size INTEGER,
    market_size_billion REAL,
    product_traction_users INTEGER,
    burn_rate_million REAL,
    revenue_million REAL,
    investor_type TEXT,
    sector TEXT,
    founder_background TEXT,
    outcome TEXT
);
""")

conn.commit()
print("Schema created successfully.")

# Confirm tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

Schema created successfully.
Tables in database: [('dim_startup',), ('dim_investor',), ('fact_funding_rounds',), ('ref_real_unicorns',), ('sqlite_sequence',), ('supplementary_funding_events',), ('ml_training_set',)]


In [5]:
# ===== CELL 5: LOAD DATA =====

# Rename df_unicorns_real columns to match schema before loading
df_unicorns_real_load = df_unicorns_real.rename(columns={
    'Company': 'company',
    'Valuation ($B)': 'valuation_billion_usd',
    'Date Joined': 'date_joined',
    'Country': 'country',
    'City': 'city',
    'Industry': 'industry',
    'Select Investors': 'select_investors'
})

df_startups.to_sql('dim_startup', conn, if_exists='append', index=False)
df_investors.to_sql('dim_investor', conn, if_exists='append', index=False)
df_funding.to_sql('fact_funding_rounds', conn, if_exists='append', index=False)
df_unicorns_real_load[['company','valuation_billion_usd','date_joined','country','city','industry','select_investors']].to_sql(
    'ref_real_unicorns', conn, if_exists='append', index=False)
df_supplementary.to_sql('supplementary_funding_events', conn, if_exists='append', index=False)
df_ml_training.to_sql('ml_training_set', conn, if_exists='append', index=False)

conn.commit()
print("All data loaded into SQLite.")

All data loaded into SQLite.


In [6]:
# ===== CELL 6: VALIDATE LOAD =====
tables = ['dim_startup', 'dim_investor', 'fact_funding_rounds', 
          'ref_real_unicorns', 'supplementary_funding_events', 'ml_training_set']

for t in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {count} rows")

dim_startup: 1747 rows
dim_investor: 73 rows
fact_funding_rounds: 2818 rows
ref_real_unicorns: 1360 rows
supplementary_funding_events: 4847 rows
ml_training_set: 100000 rows


In [7]:
# ===== CELL 7: CONFIRM DB FILE ON DISK =====
import os
print(f"Database file size: {os.path.getsize(DB_PATH) / 1e6:.2f} MB")
print(f"Location: {os.path.abspath(DB_PATH)}")

Database file size: 9.49 MB
Location: /Users/khushithakur/Analytics/Startup Investment Intelligence Project/data/startup_intelligence.db


In [8]:
# ===== CELL 8: FIRST REAL QUERY — funding by sector and stage =====
query = """
SELECT 
    f.sector,
    f.funding_stage,
    COUNT(*) as num_rounds,
    ROUND(SUM(f.amount_million_usd), 1) as total_funding_million_usd,
    ROUND(AVG(f.amount_million_usd), 2) as avg_round_size_million_usd
FROM fact_funding_rounds f
GROUP BY f.sector, f.funding_stage
ORDER BY total_funding_million_usd DESC
LIMIT 15;
"""

result = pd.read_sql_query(query, conn)
result

,sector,funding_stage,num_rounds,total_funding_million_usd,avg_round_size_million_usd
0,Artificial Intelligence,Series B,56,1963.5,35.06
1,Fintech,Series B,41,1425.4,34.77
2,Healthcare & Biotech,Series B,35,1342.9,38.37
3,E-Commerce,Series C,11,1307.2,118.83
4,Fintech,Series C,15,1295.4,86.36
5,Healthcare & Biotech,Series C,13,1263.6,97.20
6,Artificial Intelligence,Series C,16,1229.9,76.87
7,Artificial Intelligence,Series D,6,1064.1,177.35
8,Fintech,Series A,76,1017.0,13.38
9,Artificial Intelligence,Series A,69,954.3,13.83


In [9]:
query_trend = """
SELECT 
    year,
    quarter,
    COUNT(*) as num_rounds,
    ROUND(SUM(amount_million_usd), 1) as total_funding_million_usd,
    COUNT(DISTINCT startup_id) as unique_startups_funded
FROM fact_funding_rounds
GROUP BY year, quarter
ORDER BY year, quarter;
"""
df_trend = pd.read_sql_query(query_trend, conn)
df_trend

,year,quarter,num_rounds,total_funding_million_usd,unique_startups_funded
0,2020,Q1,134,1488.1,131
1,2020,Q2,115,901.8,112
2,2020,Q3,127,2414.9,124
3,2020,Q4,127,1695.8,125
4,2021,Q1,112,1380.6,110
5,2021,Q2,103,1377.5,102
6,2021,Q3,110,1135.5,110
7,2021,Q4,101,1050.5,99
8,2022,Q1,115,1792.1,112
9,2022,Q2,116,886.6,113


In [10]:
query_investors = """
SELECT 
    i.investor_name,
    i.investor_type,
    i.hq_country,
    COUNT(f.round_id) as deals_led,
    ROUND(SUM(f.amount_million_usd), 1) as total_deployed_million_usd,
    ROUND(AVG(f.amount_million_usd), 2) as avg_check_size_million_usd
FROM dim_investor i
JOIN fact_funding_rounds f ON i.investor_id = f.investor_id
GROUP BY i.investor_id
ORDER BY total_deployed_million_usd DESC
LIMIT 15;
"""
df_investor_activity = pd.read_sql_query(query_investors, conn)
df_investor_activity

,investor_name,investor_type,hq_country,deals_led,total_deployed_million_usd,avg_check_size_million_usd
0,Sequoia Capital,Angel Investor,United States,29,999.9,34.48
1,General Catalyst,Venture Capital,United States,36,921.3,25.59
2,Y Combinator,Private Equity,United Kingdom,47,822.7,17.50
3,Rakuten Capital,Venture Capital,United Kingdom,33,778.1,23.58
4,NVIDIA NVentures,Angel Investor,India,47,751.5,15.99
5,Abu Dhabi Investment Authority,Angel Investor,Japan,50,725.6,14.51
6,Microsoft M12,Venture Capital,United States,39,718.6,18.43
7,Amazon Alexa Fund,Angel Investor,United States,45,704.3,15.65
8,GV (Google Ventures),Venture Capital,United States,37,679.6,18.37
9,B Capital Group,Venture Capital,United Kingdom,38,669.2,17.61


In [11]:
query_startup_rollup = """
SELECT 
    s.startup_id,
    s.startup_name,
    s.sector,
    s.country,
    s.founded_year,
    s.revenue_status,
    COUNT(f.round_id) as total_rounds,
    ROUND(SUM(f.amount_million_usd), 2) as total_raised_million_usd,
    MAX(f.post_money_valuation_million_usd) as latest_valuation_million_usd,
    MAX(f.funding_date) as most_recent_round_date
FROM dim_startup s
LEFT JOIN fact_funding_rounds f ON s.startup_id = f.startup_id
GROUP BY s.startup_id
ORDER BY total_raised_million_usd DESC NULLS LAST
LIMIT 15;
"""
df_startup_rollup = pd.read_sql_query(query_startup_rollup, conn)
df_startup_rollup

,startup_id,startup_name,sector,country,founded_year,revenue_status,total_rounds,total_raised_million_usd,latest_valuation_million_usd,most_recent_round_date
0,S01445,XenoMind,HR Tech,Indonesia,2021,Pre-Revenue,3,394.21,2366.35,2025-02-07
1,S00194,CruxWare,E-Commerce,United Kingdom,2017,Pre-Revenue,3,369.97,2405.16,2025-07-06
2,S00695,ProtoPay,Food & AgriTech,Indonesia,2019,Pre-Revenue,3,358.78,1626.88,2025-11-20
3,S01381,Runeio,Healthcare & Biotech,Kenya,2020,Pre-Revenue,3,332.12,1365.53,2025-06-15
4,S01456,LumeAI,Fintech,Thailand,2024,Revenue Generating,3,312.09,2082.06,2026-02-18
5,S00301,RapidLabs,Healthcare & Biotech,Israel,2024,Pre-Revenue,3,302.94,1959.90,2025-06-19
6,S00652,JetX,Space Tech,United States,2022,Revenue Generating,3,302.85,1916.99,2022-11-01
7,S00596,LumeRobotics,Healthcare & Biotech,China,2020,Profitable,3,280.97,2024.01,2025-12-08
8,S01592,BoltBio,E-Commerce,Germany,2020,Pre-Revenue,3,278.22,1560.34,2025-01-04
9,S01099,LumeAnalytics,Social Media / Creator Economy,Australia,2016,Pre-Revenue,3,247.12,877.66,2025-11-08


In [12]:
query_window = """
SELECT 
    year,
    sector,
    ROUND(SUM(amount_million_usd), 1) as sector_funding_million_usd,
    RANK() OVER (PARTITION BY year ORDER BY SUM(amount_million_usd) DESC) as sector_rank
FROM fact_funding_rounds
GROUP BY year, sector
ORDER BY year, sector_rank
LIMIT 25;
"""
df_sector_rank = pd.read_sql_query(query_window, conn)
df_sector_rank

,year,sector,sector_funding_million_usd,sector_rank
0,2020,Artificial Intelligence,1429.3,1
1,2020,Healthcare & Biotech,758.0,2
2,2020,Fintech,660.2,3
3,2020,Robotics & Automation,422.3,4
4,2020,HR Tech,408.2,5
5,2020,Food & AgriTech,360.6,6
6,2020,Space Tech,336.7,7
7,2020,SaaS / Enterprise Software,327.4,8
8,2020,Logistics & Supply Chain,313.8,9
9,2020,Social Media / Creator Economy,216.3,10


In [13]:
# Check the actual distribution of total_rounds across ALL startups, not just the top 15
query_rounds_dist = """
SELECT 
    total_rounds,
    COUNT(*) as num_startups
FROM (
    SELECT s.startup_id, COUNT(f.round_id) as total_rounds
    FROM dim_startup s
    LEFT JOIN fact_funding_rounds f ON s.startup_id = f.startup_id
    GROUP BY s.startup_id
)
GROUP BY total_rounds
ORDER BY total_rounds;
"""
df_rounds_dist = pd.read_sql_query(query_rounds_dist, conn)
df_rounds_dist

,total_rounds,num_startups
0,1,956
1,2,511
2,3,280


In [14]:
query_investors = """
SELECT 
    i.investor_name,
    i.investor_type,
    i.hq_country,
    COUNT(f.round_id) as deals_led,
    ROUND(SUM(f.amount_million_usd), 1) as total_deployed_million_usd,
    ROUND(AVG(f.amount_million_usd), 2) as avg_check_size_million_usd
FROM dim_investor i
JOIN fact_funding_rounds f ON i.investor_id = f.investor_id
GROUP BY i.investor_id
ORDER BY total_deployed_million_usd DESC
LIMIT 15;
"""
df_investor_activity = pd.read_sql_query(query_investors, conn)
df_investor_activity

,investor_name,investor_type,hq_country,deals_led,total_deployed_million_usd,avg_check_size_million_usd
0,Sequoia Capital,Angel Investor,United States,29,999.9,34.48
1,General Catalyst,Venture Capital,United States,36,921.3,25.59
2,Y Combinator,Private Equity,United Kingdom,47,822.7,17.50
3,Rakuten Capital,Venture Capital,United Kingdom,33,778.1,23.58
4,NVIDIA NVentures,Angel Investor,India,47,751.5,15.99
5,Abu Dhabi Investment Authority,Angel Investor,Japan,50,725.6,14.51
6,Microsoft M12,Venture Capital,United States,39,718.6,18.43
7,Amazon Alexa Fund,Angel Investor,United States,45,704.3,15.65
8,GV (Google Ventures),Venture Capital,United States,37,679.6,18.37
9,B Capital Group,Venture Capital,United Kingdom,38,669.2,17.61


In [15]:
# Check a few more well-known names against their real known HQs
known_check = df_investors[df_investors['investor_name'].isin(
    ['Y Combinator', 'Sequoia Capital', 'Andreessen Horowitz (a16z)', 'Tiger Global']
)][['investor_name', 'hq_country']]
known_check

,investor_name,hq_country
0,Sequoia Capital,United States
1,Andreessen Horowitz (a16z),United States
2,Tiger Global,United States
6,Y Combinator,United Kingdom


In [16]:
# Get a fuller picture: how many total rows, and roughly how many look wrong at a glance
print(f"Total investors: {len(df_investors)}")
df_investors[['investor_name', 'hq_country']].head(20)

Total investors: 73


,investor_name,hq_country
0,Sequoia Capital,United States
1,Andreessen Horowitz (a16z),United States
2,Tiger Global,United States
3,Accel Partners,United States
4,SoftBank Vision Fund,India
5,Lightspeed Venture Partners,India
6,Y Combinator,United Kingdom
7,Benchmark,United States
8,General Catalyst,United States
9,Insight Partners,United States


In [17]:
query_cohort = """
WITH cohort_startups AS (
    SELECT startup_id, startup_name, founded_year, sector
    FROM dim_startup
),
cohort_funding AS (
    SELECT 
        cs.founded_year,
        COUNT(DISTINCT cs.startup_id) as startups_in_cohort,
        COUNT(f.round_id) as total_rounds_raised,
        ROUND(SUM(f.amount_million_usd), 1) as total_raised_million_usd,
        ROUND(AVG(f.amount_million_usd), 2) as avg_round_size_million_usd
    FROM cohort_startups cs
    LEFT JOIN fact_funding_rounds f ON cs.startup_id = f.startup_id
    GROUP BY cs.founded_year
)
SELECT 
    founded_year,
    startups_in_cohort,
    total_rounds_raised,
    total_raised_million_usd,
    ROUND(total_raised_million_usd / startups_in_cohort, 2) as avg_raised_per_startup_million_usd
FROM cohort_funding
ORDER BY founded_year;
"""
df_cohort = pd.read_sql_query(query_cohort, conn)
df_cohort

,founded_year,startups_in_cohort,total_rounds_raised,total_raised_million_usd,avg_raised_per_startup_million_usd
0,2015,63,101,1120.2,17.78
1,2016,60,94,1340.6,22.34
2,2017,95,161,1711.7,18.02
3,2018,126,197,2395.5,19.01
4,2019,138,216,2216.7,16.06
5,2020,196,314,3803.3,19.40
6,2021,269,414,4985.4,18.53
7,2022,316,521,5853.9,18.52
8,2023,264,449,5488.0,20.79
9,2024,220,351,4609.9,20.95
